# Análise de Parâmetros CUSUM (K e H)

Este notebook tem como objetivo realizar uma análise empírica para determinar os valores ideais para os parâmetros `k` e `h` do algoritmo CUSUM, que por sua vez definem `K` e `H`.

**Metodologia:**
1. **Carregar Dados:** Carregar os dados processados da camada de staging (`app/data/staging/`).
2. **Fase I - Calibração:** Selecionar um período de dados estável para calcular a média (μ₀) e o desvio padrão (σ₀) para cada canal horário de cada dispositivo.
3. **Fase II - Simulação e Análise:** Aplicar o algoritmo CUSUM em um período de teste, experimentando diferentes valores de `k` e `h` para encontrar um equilíbrio entre a minimização de falsos alarmes (ARL₀ longo) e a detecção rápida de anomalias reais (ARL₁ curto).
4. **Conclusão:** Documentar os valores recomendados e a justificativa.

In [1]:
import pandas as pd
import duckdb
import os
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## 1. Carregar Dados

Vamos carregar os dados da camada de staging. Como os dados estão particionados por `event_date`, podemos usar o DuckDB para ler todos os arquivos Parquet de forma eficiente.

In [2]:
DATA_PATH = '../data/staging/'

con = duckdb.connect(database=':memory:', read_only=False)
df = con.execute(f"""SELECT *
FROM read_parquet('{DATA_PATH}/**/*.parquet', hive_partitioning=1)
""").fetchdf()
con.close()

# Converter event_time para datetime e ordenar
df['event_time'] = pd.to_datetime(df['event_time'])
df = df.sort_values(by='event_time').reset_index(drop=True)

print(f'Dados carregados com {df.shape[0]} registros.')
print(f"Período dos dados: {df['event_time'].min()} a {df['event_time'].max()}")
df.head()

Dados carregados com 35999 registros.
Período dos dados: 2025-04-18 20:46:01.179000 a 2025-06-17 10:29:31


,code,value,device_id,ingestion_timestamp_utc,ingested_by,filename,event_time,device_name,event_date
0,cur_voltage,1225,ebb50554f386a6d20fvbwv,2025-04-25T23:35:18.579808+00:00,tuya_log_ingestion_script,/Users/igorcleto/Documents/UFMG/PFC/PFC_1_igor...,2025-04-18 20:46:01.179,Fita de LED,2025-04-18
1,cur_voltage,1231,eb47e52ca43f9bc6349k2v,2025-04-25T23:35:18.579808+00:00,tuya_log_ingestion_script,/Users/igorcleto/Documents/UFMG/PFC/PFC_1_igor...,2025-04-18 20:46:04.700,Repelente,2025-04-18
2,add_ele,6,ebb1ae44e3bb8945c7colh,2025-04-25T23:35:18.579808+00:00,tuya_log_ingestion_script,/Users/igorcleto/Documents/UFMG/PFC/PFC_1_igor...,2025-04-18 21:01:43.000,Hack Sala,2025-04-18
3,cur_voltage,1248,ebf51d0d4012b176f2xhna,2025-04-25T23:35:18.579808+00:00,tuya_log_ingestion_script,/Users/igorcleto/Documents/UFMG/PFC/PFC_1_igor...,2025-04-18 21:02:51.723,Ventilador do quarto,2025-04-18
4,cur_voltage,1231,ebf51d0d4012b176f2xhna,2025-04-25T23:35:18.579808+00:00,tuya_log_ingestion_script,/Users/igorcleto/Documents/UFMG/PFC/PFC_1_igor...,2025-04-18 21:11:26.169,Ventilador do quarto,2025-04-18


## 2. Fase I - Calibração

Nesta fase, vamos calcular os parâmetros de controle (média e desvio padrão) para um processo considerado "em controle". Usaremos os dados da 'Geladeira', que possui um ciclo de consumo relativamente estável.

Vamos usar as primeiras semanas de dados como período de calibração.

In [3]:
# Selecionar o dispositivo e o período de calibração
device_to_analyze = 'Hack Sala'
calibration_end_date = '2025-05-15' # Corrigido para uma data com dados existentes

# Filtrar apenas os dados de potência ('cur_power')
df_power = df[df['code'] == 'cur_power'].copy()
df_power['power'] = df_power['value'].astype(float) / 10.0

df_device = df_power[df_power['device_name'] == device_to_analyze].copy()
df_calibration = df_device[df_device['event_time'] <= calibration_end_date].copy()
df_monitoring = df_device[df_device['event_time'] > calibration_end_date].copy()

# Extrair o canal horário
df_calibration['hour_channel'] = df_calibration['event_time'].dt.hour

# Calcular média (mu_0) e desvio padrão (sigma_0) para cada canal horário
control_params = df_calibration.groupby('hour_channel')['power'].agg(['mean', 'std']).reset_index()
control_params = control_params.rename(columns={'mean': 'mu_0', 'std': 'sigma_0'})

# Lidar com canais que possam não ter variabilidade (std=0)
control_params['sigma_0'] = control_params['sigma_0'].replace(0, 0.01) # Substituir por um valor pequeno para evitar divisão por zero

print(f"Parâmetros de controle calculados para '{device_to_analyze}' usando {df_calibration.shape[0]} registros.")
control_params.head()

Parâmetros de controle calculados para 'Hack Sala' usando 1304 registros.


,hour_channel,mu_0,sigma_0
0,0,13.531250,2.475745
1,1,13.853333,2.714296
2,2,13.335000,2.517366
3,3,14.169231,2.835596
4,4,13.494118,2.359945


## 3. Fase II - Simulação e Análise

Agora, vamos aplicar o algoritmo CUSUM nos dados de monitoramento (`df_monitoring`) usando os parâmetros da Fase I. Criaremos uma função para executar o CUSUM e testaremos diferentes combinações de `k` e `h` para avaliar a sensibilidade.

In [4]:
def run_cusum_simulation(df_monitor, control_params, k, h):
    """Aplica o algoritmo CUSUM a um dataframe de monitoramento."""
    df_monitor = df_monitor.copy()
    # A coluna 'power' já foi calculada, então não precisamos dividir por 10 novamente.
    df_monitor['hour_channel'] = df_monitor['event_time'].dt.hour
    
    # Juntar os parâmetros de controle
    df_merged = pd.merge(df_monitor, control_params, on='hour_channel', how='left')
    
    # Inicializar colunas CUSUM
    df_merged['K'] = k * df_merged['sigma_0']
    df_merged['H'] = h * df_merged['sigma_0']
    df_merged['Sh'] = 0.0
    df_merged['Sl'] = 0.0
    
    # Iterar para calcular as somas acumuladas
    for i in range(1, len(df_merged)):
        # CUSUM Superior (para detectar aumentos)
        sh_prev = df_merged.loc[i-1, 'Sh']
        x_i = df_merged.loc[i, 'power']
        mu_0 = df_merged.loc[i, 'mu_0']
        K = df_merged.loc[i, 'K']
        df_merged.loc[i, 'Sh'] = max(0, sh_prev + (x_i - (mu_0 + K)))
        
        # CUSUM Inferior (para detectar reduções)
        sl_prev = df_merged.loc[i-1, 'Sl']
        df_merged.loc[i, 'Sl'] = max(0, sl_prev + ((mu_0 - K) - x_i))
        
    # Identificar alarmes
    df_merged['alarm'] = (df_merged['Sh'] > df_merged['H']) | (df_merged['Sl'] > df_merged['H'])
    
    return df_merged

# --- Simulação 1: k=0.5, h=5 (Padrão da Monografia) ---
df_sim1 = run_cusum_simulation(df_monitoring, control_params, k=0.5, h=5)
alarms1_count = df_sim1['alarm'].sum()

# --- Simulação 2: k=0.75, h=4 (Menos sensível) ---
df_sim2 = run_cusum_simulation(df_monitoring, control_params, k=0.75, h=4)
alarms2_count = df_sim2['alarm'].sum()

print(f"Simulação 1 (k=0.5, h=5): {alarms1_count} alarmes")
print(f"Simulação 2 (k=0.75, h=4): {alarms2_count} alarmes")

Simulação 1 (k=0.5, h=5): 3637 alarmes
Simulação 2 (k=0.75, h=4): 2609 alarmes


### Visualização dos Resultados

Vamos plotar os resultados para comparar visualmente o comportamento das duas simulações.

In [5]:
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, 
                    subplot_titles=('Consumo de Potência (W)', 'CUSUM (k=0.5, h=5)', 'CUSUM (k=0.75, h=4)'))

# Gráfico de Potência
fig.add_trace(go.Scatter(x=df_sim1['event_time'], y=df_sim1['power'], mode='lines', name='Potência'), row=1, col=1)

# Gráfico Simulação 1
fig.add_trace(go.Scatter(x=df_sim1['event_time'], y=df_sim1['Sh'], mode='lines', name='Sh (k=0.5, h=5)'), row=2, col=1)
fig.add_trace(go.Scatter(x=df_sim1['event_time'], y=df_sim1['Sl'], mode='lines', name='Sl (k=0.5, h=5)'), row=2, col=1)
fig.add_trace(go.Scatter(x=df_sim1['event_time'], y=df_sim1['H'], mode='lines', name='Limite H', line=dict(dash='dash', color='red')), row=2, col=1)
fig.add_trace(go.Scatter(x=df_sim1[df_sim1['alarm']]['event_time'], y=df_sim1[df_sim1['alarm']]['Sh'], mode='markers', name='Alarme Sh', marker=dict(color='red', size=8)), row=2, col=1)

# Gráfico Simulação 2
fig.add_trace(go.Scatter(x=df_sim2['event_time'], y=df_sim2['Sh'], mode='lines', name='Sh (k=0.75, h=4)'), row=3, col=1)
fig.add_trace(go.Scatter(x=df_sim2['event_time'], y=df_sim2['Sl'], mode='lines', name='Sl (k=0.75, h=4)'), row=3, col=1)
fig.add_trace(go.Scatter(x=df_sim2['event_time'], y=df_sim2['H'], mode='lines', name='Limite H', line=dict(dash='dash', color='red')), row=3, col=1)
fig.add_trace(go.Scatter(x=df_sim2[df_sim2['alarm']]['event_time'], y=df_sim2[df_sim2['alarm']]['Sh'], mode='markers', name='Alarme Sh', marker=dict(color='red', size=8)), row=3, col=1)

fig.update_layout(height=800, title_text=f"Análise CUSUM para '{device_to_analyze}'")
fig.show()